In [1]:
import pandas as pd

df = pd.read_csv("../data/gdelt_clean.csv", keep_default_na=False)

print("Total articles:", len(df))
print("\nColumns:")
print(df.columns.tolist())

Total articles: 15395

Columns:
['article_id', 'title', 'publication_date', 'country', 'country_code', 'continent', 'topic', 'publisher', 'article_url', 'all_countries', 'all_country_codes', 'all_continents', 'primary_topic', 'original_themes']


In [2]:
# Find articles containing the suspicious GDELT theme
telecom_articles = df[
    df["original_themes"].str.contains(
        "CRISISLEX_C08_TELECOM",
        case=False,
        na=False
    )
].copy()

# Check how many are classified as Technology
telecom_technology = telecom_articles[
    telecom_articles["topic"].str.split(";").apply(
        lambda topics: "Technology" in topics
    )
]

print("Articles containing CRISISLEX_C08_TELECOM:", len(telecom_articles))
print("Of these, articles labeled Technology:", len(telecom_technology))

# Examine a sample
telecom_technology[
    ["title", "topic", "primary_topic"]
].head(20)

Articles containing CRISISLEX_C08_TELECOM: 9
Of these, articles labeled Technology: 2


,title,topic,primary_topic
4625,Uldale road set to close for five days this week,Technology,Technology
5527,Bickington road closures planned for major gas...,Technology,Technology


In [3]:
# Select articles classified primarily as Technology
technology_articles = df[
    df["primary_topic"] == "Technology"
].copy()

print("Total primary Technology articles:", len(technology_articles))

# Random sample for inspection
technology_articles[
    ["title", "topic", "original_themes"]
].sample(
    n=min(20, len(technology_articles)),
    random_state=42
)

Total primary Technology articles: 560


,title,topic,original_themes
11677,Lectra takes AI upstream with Apogy,Technology; Business and Economy; Education; E...,SOC_EMERGINGTECH;EPU_ECONOMY_HISTORIC;TAX_FNCA...
9737,Innovation competition drives young researcher...,Technology; Science; Business and Economy; Edu...,NATURAL_DISASTER;NATURAL_DISASTER_FLOOD;CRISIS...
4657,M.A. Rangoonwala College Launches Abeda Inamda...,Technology; Education; Business and Economy; E...,TAX_WORLDLANGUAGES;TAX_WORLDLANGUAGES_MAHARASH...
2356,'Machines are collaborators' at this new museu...,Technology; Society and Culture,
9463,US government to use AI virtual agents as part...,Technology; Business and Economy; Politics and...,GENERAL_GOVERNMENT;EPU_POLICY;EPU_POLICY_GOVER...
3488,TRAI launches revamped MyCall app to boost con...,Technology; Environment and Climate,EPU_POLICY;EPU_POLICY_REGULATORY;WB_678_DIGITA...
7432,Cary approves 18-month moratorium on data cent...,Technology; Business and Economy; Environment ...,TAX_FNCACT;TAX_FNCACT_LEADER;WB_137_WATER;WB_1...
8364,Land of Sky distributes over 100 laptops in Pa...,Technology; Business and Economy; Education,EDUCATION;SOC_POINTSOFINTEREST;SOC_POINTSOFINT...
2607,What Nasscom wants changed in the Supreme Cour...,Technology; Business and Economy; Conflict and...,USPEC_POLICY1;EPU_ECONOMY_HISTORIC;USPEC_POLIT...
13936,Cooperatives urged to focus on data for more e...,Technology; Business and Economy; Conflict and...,TAX_WORLDFISH;TAX_WORLDFISH_FISH;TAX_ECON_PRIC...


In [4]:
pd.set_option("display.max_colwidth", None)

questionable_ids = [13936, 15140, 5550, 4657]

df.loc[
    questionable_ids,
    ["title", "topic", "primary_topic", "original_themes"]
]

,title,topic,primary_topic,original_themes
13936,Cooperatives urged to focus on data for more efficient digital transformation,Technology; Business and Economy; Conflict and Security; Environment and Climate; Science,Technology,TAX_WORLDFISH;TAX_WORLDFISH_FISH;TAX_ECON_PRICE;TAX_WORLDMAMMALS;TAX_WORLDMAMMALS_HUMAN;UNGP_FORESTS_RIVERS_OCEANS;WB_841_JUSTICE_SYSTEM_ADMINISTRATION;WB_696_PUBLIC_SECTOR_MANAGEMENT;WB_840_JUSTICE;WB_1211_HUMAN_RESOURCES;TAX_FNCACT;TAX_FNCACT_CHAIR
15140,Innovation. Connection. Success: CCE 2027 Welcomes the World to Shanghai,Technology; Business and Economy; Conflict and Security; Education; Environment and Climate; Health; Politics and Government; Science; Society and Culture,Technology,WB_1458_HEALTH_PROMOTION_AND_DISEASE_PREVENTION;WB_1462_WATER_SANITATION_AND_HYGIENE;WB_635_PUBLIC_HEALTH;WB_621_HEALTH_NUTRITION_AND_POPULATION;EPU_ECONOMY_HISTORIC;SOC_INNOVATION;WB_566_ENVIRONMENT_AND_NATURAL_RESOURCES;WB_590_ECOSYSTEMS;UNGP_FORES
5550,Monaco tables new law to overhaul rules for crypto-asset firms,Technology; Business and Economy; Conflict and Security; Politics and Government,Technology,GENERAL_GOVERNMENT;EPU_POLICY;EPU_POLICY_GOVERNMENT;LEGISLATION;EPU_POLICY_LAW;USPEC_POLICY1;EPU_POLICY_REGULATION;EPU_CATS_REGULATION;WB_775_TRADE_POLICY_AND_INTEGRATION;WB_698_TRADE;WB_768_INTERNATIONAL_STANDARDS_AND_TECHNICAL_REGULATIONS;WB_778_NO
4657,M.A. Rangoonwala College Launches Abeda Inamdar Centre of Implantology and Digital Dentistry in Pune,Technology; Education; Business and Economy; Environment and Climate; Health; Politics and Government,Technology,TAX_WORLDLANGUAGES;TAX_WORLDLANGUAGES_MAHARASHTRA;EDUCATION;SOC_POINTSOFINTEREST;SOC_POINTSOFINTEREST_COLLEGE;WB_470_EDUCATION;CRISISLEX_C03_WELLBEING_HEALTH;TAX_FNCACT;TAX_FNCACT_STUDENTS;LEADER;TAX_FNCACT_PRESIDENT;USPEC_POLITICS_GENERAL1;MEDICAL;U


In [5]:
# Count the number of assigned topics per article
df["topic_count"] = df["topic"].apply(
    lambda x: len([t.strip() for t in x.split(";") if t.strip()])
)

# Distribution of topic counts
topic_distribution = (
    df["topic_count"]
    .value_counts()
    .sort_index()
)

print("Articles by number of topics:")
print(topic_distribution)

print("\nArticles assigned all 9 topics:")
print((df["topic_count"] == 9).sum())

print("\nArticles assigned 7 or more topics:")
print((df["topic_count"] >= 7).sum())

Articles by number of topics:
topic_count
1    3322
2    2012
3    2424
4    2472
5    2195
6    1573
7     938
8     364
9      95
Name: count, dtype: int64

Articles assigned all 9 topics:
95

Articles assigned 7 or more topics:
1397


In [6]:
# Select articles assigned all nine topics
all_topic_articles = df[df["topic_count"] == 9]

# Display 15 random article titles
all_topic_articles[
    ["title", "primary_topic"]
].sample(
    n=15,
    random_state=42
)

,title,primary_topic
11418,"CNN, MS NOW and Politico sue Trump. And, Paramount settles in massive merger lawsuit",Business and Economy
4316,Prime Minister Narendra Modi addresses the 57th Convocation Ceremony of IIT Delhi - India Education | Latest Education News | Global Educational News,Politics and Government
12542,UK Labour Party holds its first conference under new leader Burnham,Business and Economy
12715,San Francisco's car crime has plunged. How much credit does Flock deserve?,Conflict and Security
122,'The Trojan Teddy Bear': The promise and peril of childhood in the age of AI,Technology
4788,ADC Therapeutics Reports Second Quarter 2026 Financial Results and Provides Operational Updates,Business and Economy
5526,Call for Proposals: Small Grants Programme for Civil Society Organizations (Uzbekistan),Society and Culture
6493,"Helus Pharma Appoints Regina Donohue as Chief People Officer and Suresh Durgam, M.D., to its Scientific Advisory Board",Science
1229,In Whose Service? - Nassau Weekly,Business and Economy
6739,8 Horror Books That Are 10/10 Masterpieces,Business and Economy


In [7]:
# Examine the themes behind three suspicious articles
sample_ids = [6739, 8154, 7240]

for idx in sample_ids:
    article = df.loc[idx]

    print("=" * 70)
    print("TITLE:", article["title"])
    print("TOPICS:", article["topic"])
    print("PRIMARY:", article["primary_topic"])
    print("\nGDELT THEMES:")

    themes = article["original_themes"].split(";")

    for theme in themes:
        if theme.strip():
            print(" -", theme)

    print()

TITLE: 8 Horror Books That Are 10/10 Masterpieces
TOPICS: Business and Economy; Conflict and Security; Education; Environment and Climate; Health; Politics and Government; Science; Society and Culture; Technology
PRIMARY: Business and Economy

GDELT THEMES:
 - TAX_DISEASE
 - TAX_DISEASE_FEVER
 - EPU_CATS_MIGRATION_FEAR_FEAR
 - TAX_ETHNICITY
 - TAX_ETHNICITY_JAPANESE
 - TAX_WORLDLANGUAGES
 - TAX_WORLDLANGUAGES_JAPANESE
 - TAX_WORLDLANGUAGES_MANGA
 - KILL
 - TAX_FNCACT
 - TAX_FNCACT_CITIZENS
 - EPU_CATS_NATIONAL_SECURITY
 - TAX_FNCACT_KING
 - TAX

TITLE: Orlando Sentinel 2026-27 Arts Season Preview
TOPICS: Society and Culture; Business and Economy; Conflict and Security; Education; Environment and Climate; Health; Politics and Government; Science; Technology
PRIMARY: Society and Culture

GDELT THEMES:
 - TAX_FNCACT
 - TAX_FNCACT_CURATOR
 - MANMADE_DISASTER_IMPLIED
 - UNGP_FORESTS_RIVERS_OCEANS
 - EDUCATION
 - SOC_POINTSOFINTEREST
 - SOC_POINTSOFINTEREST_SCHOOL
 - TAX_FNCACT_ARTISTS
 - WB

In [16]:
from pathlib import Path

print("Current folder:", Path.cwd())

for file in Path.cwd().parent.rglob("data_preparation*.ipynb"):
    print(file)

Current folder: /Users/nova/Documents/hackathon_project/Phase_2/notebooks
/Users/nova/Documents/hackathon_project/Phase_2/notebooks/data_preparation.ipynb


In [17]:
import re

# Load the original classification patterns from your teammate's notebook
import json

with open("data_preparation.ipynb", "r") as f:
    notebook = json.load(f)

classification_cell = next(
    "".join(cell["source"])
    for cell in notebook["cells"]
    if "TITLE_PATTERNS = {" in "".join(cell.get("source", []))
)

# Extract only the TITLE_PATTERNS dictionary
pattern_code = classification_cell.split(
    "# 2. Strict word-bounded"
)[0]

namespace = {}
exec(pattern_code, namespace)

TITLE_PATTERNS = namespace["TITLE_PATTERNS"]

# Identify headline-supported topics
def get_title_topics(title):
    return [
        category
        for category, pattern in TITLE_PATTERNS.items()
        if re.search(pattern, str(title).lower())
    ]

df["title_topics"] = df["title"].apply(get_title_topics)

df["existing_topics"] = df["topic"].apply(
    lambda x: [t.strip() for t in x.split(";") if t.strip()]
)

df["theme_only_topics"] = df.apply(
    lambda row: [
        topic for topic in row["existing_topics"]
        if topic not in row["title_topics"]
    ],
    axis=1
)

print("Total articles:", len(df))
print(
    "Articles with no headline topic match:",
    (df["title_topics"].apply(len) == 0).sum()
)

print(
    "Articles with at least one theme-only topic:",
    (df["theme_only_topics"].apply(len) > 0).sum()
)

Total articles: 15395
Articles with no headline topic match: 9844
Articles with at least one theme-only topic: 14755


In [18]:
# Articles where at least one headline topic was detected
matched_articles = df[
    df["title_topics"].apply(len) > 0
].copy()

# Check whether all headline-supported topics exist in the final labels
matched_articles["headline_topics_preserved"] = matched_articles.apply(
    lambda row: set(row["title_topics"]).issubset(
        set(row["existing_topics"])
    ),
    axis=1
)

print("Articles with headline topic matches:", len(matched_articles))

print(
    "Articles preserving all headline topics:",
    matched_articles["headline_topics_preserved"].sum()
)

print(
    "Articles missing headline-supported topics:",
    (~matched_articles["headline_topics_preserved"]).sum()
)

Articles with headline topic matches: 5551
Articles preserving all headline topics: 5551
Articles missing headline-supported topics: 0


In [19]:
# Articles with exactly one headline-matched topic
# but seven or more assigned topics

suspicious_articles = df[
    (df["title_topics"].apply(len) == 1) &
    (df["topic_count"] >= 7)
].copy()

print("Potentially overclassified articles:", len(suspicious_articles))

# Examine a reproducible random sample
suspicious_articles[
    ["title", "title_topics", "theme_only_topics", "topic_count"]
].sample(
    n=min(15, len(suspicious_articles)),
    random_state=42
)

Potentially overclassified articles: 592


,title,title_topics,theme_only_topics,topic_count
13390,"UH Mānoa grad students honored for character, integrity and commitment to harmony",[Education],"[Business and Economy, Conflict and Security, Environment and Climate, Politics and Government, Society and Culture, Technology]",7
15263,2 US economists win Nobel prize | The Arkansas Democrat-Gazette - Arkansas' Best News Source,[Politics and Government],"[Business and Economy, Education, Environment and Climate, Health, Society and Culture, Technology]",7
5960,Texas Capital's TXS and OILT to Become First Primary Listings on the Texas Stock Exchange,[Business and Economy],"[Conflict and Security, Environment and Climate, Politics and Government, Science, Society and Culture, Technology]",7
5311,Eurovision music contest will take place in Bulgaria's Black Sea city of Burgas next year,[Society and Culture],"[Business and Economy, Conflict and Security, Environment and Climate, Health, Politics and Government, Technology]",7
14351,Henri B. Kagan and Kenso Soai win Nobel Prize in chemistry,[Science],"[Business and Economy, Education, Environment and Climate, Health, Politics and Government, Society and Culture]",7
960,The Local's guide to Sweden's political parties: The Green Party,[Politics and Government],"[Business and Economy, Conflict and Security, Education, Environment and Climate, Health, Science, Society and Culture]",8
14130,Chemistry Nobel Prize awarded to duo for research solving 'mirror image' mystery,[Science],"[Business and Economy, Conflict and Security, Education, Environment and Climate, Health, Politics and Government, Society and Culture]",8
9790,Market diversification strategies used by European gambling groups,[Business and Economy],"[Conflict and Security, Environment and Climate, Health, Politics and Government, Science, Technology]",7
8146,India's prime minister is wooing the GenZ while cracking down on them,[Politics and Government],"[Business and Economy, Conflict and Security, Education, Environment and Climate, Science, Society and Culture, Technology]",8
9479,BCCL and OpenAI partner to advance discovery and publishing experiences,[Science],"[Business and Economy, Conflict and Security, Environment and Climate, Politics and Government, Society and Culture, Technology]",7


In [20]:
from collections import Counter

# Count additional topics among the 592 potentially overclassified articles
extra_topic_counts = Counter(
    topic
    for topics in suspicious_articles["theme_only_topics"]
    for topic in topics
)

extra_topic_df = pd.DataFrame(
    extra_topic_counts.items(),
    columns=["Topic", "Articles"]
).sort_values("Articles", ascending=False)

extra_topic_df["Percentage"] = (
    extra_topic_df["Articles"] / len(suspicious_articles) * 100
).round(1)

display(extra_topic_df)

,Topic,Articles,Percentage
1,Environment and Climate,515,87.0
3,Politics and Government,504,85.1
0,Business and Economy,484,81.8
6,Conflict and Security,448,75.7
2,Health,435,73.5
4,Society and Culture,433,73.1
7,Education,384,64.9
5,Technology,360,60.8
8,Science,240,40.5


In [21]:
# Extract the original THEME_PATTERNS dictionary
# from your teammate's data-preparation notebook

import ast

tree = ast.parse(classification_cell)

for node in tree.body:
    if isinstance(node, ast.Assign):
        if any(
            isinstance(target, ast.Name)
            and target.id == "THEME_PATTERNS"
            for target in node.targets
        ):
            THEME_PATTERNS = ast.literal_eval(node.value)
            break

for category, patterns in THEME_PATTERNS.items():
    print(f"\n{category}")
    print(patterns)


Conflict and Security
(?:^|_)(?:WAR|MILITARY|TERROR|TERRORISM|KILL|KILLING|KILLED|ATTACK|ATTACKS|SECURITY|CRIME|WEAPON|WEAPONS|ARREST|ARRESTS|PROTEST|PROTESTS|SEIGE|BLOCKADE|ARMEDCONFLICT|INSURGENCY|POLICE|JUSTICE|COURT|EXTREMISM|VIOLENCE|DEFENSE|FIREARM|FIREARMS|MURDER)(?:_|$)

Politics and Government
(?:^|_)(?:POLITICS|POLITICAL|ELECTION|ELECTIONS|LEADER|LEADERS|GOV|PRESIDENT|MINISTER|PRIME_MINISTER|LEGISLATION|CONSTITUTIONAL|DEMOCRACY|CORRUPTION|PARLIAMENT|SENATE|CONGRESS|DIPLOMACY|TREATY|REGULATION|SOVEREIGNTY)(?:_|$)

Business and Economy
(?:^|_)(?:ECON|ECONOMIC|ECONOMY|BUSINESS|TRADE|FINANCE|FINANCIAL|MARKET|MARKETS|STOCK|STOCKS|INFLATION|BANK|BANKING|INVESTMENT|INVESTOR|COMMERCE|EMPLOYMENT|UNEMPLOYMENT|LABOR|TARIFF|TARIFFS|CURRENCY|DEBT|REVENUE|AGRICULTURE)(?:_|$)

Technology
(?:^|_)(?:TECH|TECHNOLOGY|CYBER|INTERNET|SOFTWARE|ARTIFICIAL_INTELLIGENCE|INNOVATION|TELECOM|MOBILE|COMPUTER|COMPUTERS|DIGITAL|ALGORITHM|ROBOTICS|AUTOMATION)(?:_|$)

Health
(?:^|_)(?:HEALTH|MEDICAL|DISEASE

In [22]:
# Flag potentially overclassified articles

df["needs_topic_review"] = df.apply(
    lambda row: (
        row["topic_count"] >= 7
        and len(row["theme_only_topics"]) >= 5
    ),
    axis=1
)

print(
    "Articles flagged for topic review:",
    df["needs_topic_review"].sum()
)

print(
    "Percentage of dataset:",
    round(df["needs_topic_review"].mean() * 100, 2),
    "%"
)

# Examine a sample
display(
    df[df["needs_topic_review"]][
        ["title", "title_topics", "theme_only_topics"]
    ].sample(
        n=10,
        random_state=42
    )
)

Articles flagged for topic review: 1388
Percentage of dataset: 9.02 %


,title,title_topics,theme_only_topics
7341,Six-Year-Old Aditya and the Politics of a New Generation,[Politics and Government],"[Business and Economy, Conflict and Security, Education, Environment and Climate, Society and Culture, Technology]"
2413,Hotter winds threaten to fan flames of Europe's wildfires,[],"[Business and Economy, Conflict and Security, Education, Environment and Climate, Health, Politics and Government, Society and Culture]"
5088,"Thomas Gallatin: Hating Capitalism, but Loving Free Enterprise?",[],"[Business and Economy, Education, Environment and Climate, Health, Politics and Government, Science, Technology]"
7596,Fourteen Babies and the Republic of Pakistani Privilege,[],"[Business and Economy, Conflict and Security, Education, Environment and Climate, Health, Politics and Government, Society and Culture]"
14441,"3 years on, Israelis grapple with reports Netanyahu was warned before Oct. 7 attacks",[Conflict and Security],"[Education, Environment and Climate, Politics and Government, Science, Society and Culture, Technology]"
2805,"Pakistan GSP+ status under threat over rights, governance gaps",[],"[Business and Economy, Conflict and Security, Education, Environment and Climate, Politics and Government, Society and Culture, Technology]"
11191,"Trump set to address UN amid war, AI anxiety and questions about world body's future","[Conflict and Security, Technology]","[Business and Economy, Environment and Climate, Health, Politics and Government, Society and Culture]"
6072,Metals Creek Resources Corp.: Metals Creek and Benton Complete Hydrogen-Helium Soil Gas Sampling at Smoking Gun and Parson's Pond in Newfoundland and Acquires Addition Ground,[Conflict and Security],"[Business and Economy, Environment and Climate, Health, Politics and Government, Society and Culture, Technology]"
14181,KITS Eyecare Reports Preliminary Third Quarter 2026 Results,[],"[Business and Economy, Conflict and Security, Environment and Climate, Health, Politics and Government, Science, Society and Culture, Technology]"
15314,Zelenskyy says Ukraine received no intelligence from partners on incoming missile strike on Kyiv,[Conflict and Security],"[Business and Economy, Environment and Climate, Health, Politics and Government, Society and Culture, Technology]"


In [23]:
# Create a reproducible sample for manual topic review

flagged_sample = df[df["needs_topic_review"]].sample(
    n=15,
    random_state=42
)

unflagged_sample = df[~df["needs_topic_review"]].sample(
    n=15,
    random_state=42
)

review_sample = pd.concat(
    [flagged_sample, unflagged_sample],
    ignore_index=True
)

# Shuffle the combined sample
review_sample = review_sample.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Keep original labels for comparison
review_sample = review_sample[
    [
        "article_id",
        "title",
        "topic",
        "primary_topic",
        "needs_topic_review"
    ]
].copy()

# Add columns for human evaluation
review_sample["reviewed_topics"] = ""
review_sample["review_notes"] = ""

review_sample.to_csv(
    "../data/topic_review_sample.csv",
    index=False
)

print("Review sample created:", len(review_sample))
print("Saved to: data/topic_review_sample.csv")

Review sample created: 30
Saved to: data/topic_review_sample.csv


In [24]:
review_sample[
    ["title", "topic", "needs_topic_review"]
].head(5)

,title,topic,needs_topic_review
0,Bank credit surge supports high GDP growth target: SBV,Business and Economy; Conflict and Security; Environment and Climate; Politics and Government,False
1,Experts Explain How To Tell If You Have A Calcium Deficiency,Environment and Climate; Health; Politics and Government; Society and Culture,False
2,Donald Trump meets Irish president Catherine Connolly at start of two-day visit,Politics and Government; Business and Economy; Conflict and Security; Environment and Climate,False
3,Early Voting Begins Today in Miami-Dade; Broward Opens Sites Next Week,Politics and Government,False
4,KITS Eyecare Reports Preliminary Third Quarter 2026 Results,Business and Economy; Conflict and Security; Environment and Climate; Health; Politics and Government; Science; Society and Culture; Technology,True


In [25]:
# Preliminary topic review based on headlines only

preliminary_reviews = {
    0: {
        "topics": "Business and Economy",
        "notes": "Other assigned topics need verification."
    },
    1: {
        "topics": "Health",
        "notes": "Environment, Politics, and Society are not supported by the headline."
    },
    2: {
        "topics": "Politics and Government",
        "notes": "Other assigned topics need verification."
    },
    3: {
        "topics": "Politics and Government",
        "notes": "Existing classification appears appropriate."
    },
    4: {
        "topics": "Business and Economy; Health",
        "notes": "Health is plausible; several other labels need verification."
    }
}

for idx, review in preliminary_reviews.items():
    review_sample.loc[idx, "reviewed_topics"] = review["topics"]
    review_sample.loc[idx, "review_notes"] = review["notes"]

review_sample.to_csv(
    "../data/topic_review_sample.csv",
    index=False
)

print("Saved preliminary reviews for five articles.")

Saved preliminary reviews for five articles.


In [26]:
# Add a column to track verification status
review_sample["review_status"] = "Not reviewed"

# First five articles were reviewed using headlines only
review_sample.loc[0:4, "review_status"] = "Headline review only"

# Save updated review file
review_sample.to_csv(
    "../data/topic_review_sample.csv",
    index=False
)

print(review_sample["review_status"].value_counts())

review_status
Not reviewed            25
Headline review only     5
Name: count, dtype: int64


In [27]:
display(
    review_sample.loc[
        5:9,
        [
            "title",
            "topic",
            "needs_topic_review",
            "review_status"
        ]
    ]
)

,title,topic,needs_topic_review,review_status
5,Zelenskyy says Ukraine received no intelligence from partners on incoming missile strike on Kyiv,Conflict and Security; Business and Economy; Environment and Climate; Health; Politics and Government; Society and Culture; Technology,True,Not reviewed
6,OSCP open-sources a ROS 2 driver for its MK2 inertial measurement units,Business and Economy; Environment and Climate; Health,False,Not reviewed
7,Cuban artist and dissident exiled from country after 5 years in prison arrives in US,Conflict and Security; Politics and Government; Society and Culture; Technology,False,Not reviewed
8,Chemistry Nobel Prize awarded to duo for research solving 'mirror image' mystery,Science; Business and Economy; Conflict and Security; Education; Environment and Climate; Health; Politics and Government; Society and Culture,True,Not reviewed
9,Six-Year-Old Aditya and the Politics of a New Generation,Politics and Government; Business and Economy; Conflict and Security; Education; Environment and Climate; Society and Culture; Technology,True,Not reviewed


In [28]:
preliminary_reviews = {
    5: {
        "topics": "Conflict and Security; Politics and Government",
        "notes": "Other labels are not clearly supported by the headline."
    },
    6: {
        "topics": "Technology",
        "notes": "Technology is missing from existing labels. Science may also be relevant."
    },
    7: {
        "topics": "Politics and Government; Society and Culture",
        "notes": "Conflict and Security is possible. Technology needs verification."
    },
    8: {
        "topics": "Science",
        "notes": "Education is possible. Other labels need verification."
    },
    9: {
        "topics": "Politics and Government",
        "notes": "Society and Culture is possible. Other labels need verification."
    }
}

for idx, review in preliminary_reviews.items():
    review_sample.loc[idx, "reviewed_topics"] = review["topics"]
    review_sample.loc[idx, "review_notes"] = review["notes"]
    review_sample.loc[idx, "review_status"] = "Headline review only"

review_sample.to_csv(
    "../data/topic_review_sample.csv",
    index=False
)

print("Preliminary reviews saved:", 
      (review_sample["review_status"] == "Headline review only").sum())

Preliminary reviews saved: 10


In [29]:
display(
    review_sample.loc[
        10:14,
        [
            "title",
            "topic",
            "needs_topic_review",
            "review_status"
        ]
    ]
)

,title,topic,needs_topic_review,review_status
10,"3 years on, Israelis grapple with reports Netanyahu was warned before Oct. 7 attacks",Conflict and Security; Education; Environment and Climate; Politics and Government; Science; Society and Culture; Technology,True,Not reviewed
11,'Doctor ne aankhain nikal di': Uttarakhand montessary school's anti-mobile phone activity leaves kids crying and screaming in fear,Health; Technology; Education,False,Not reviewed
12,"Pakistan GSP+ status under threat over rights, governance gaps",Business and Economy; Conflict and Security; Education; Environment and Climate; Politics and Government; Society and Culture; Technology,True,Not reviewed
13,Lindela Repatriation Centre 'overwhelmed',Business and Economy; Conflict and Security; Education; Environment and Climate; Health; Politics and Government; Society and Culture; Technology,True,Not reviewed
14,"5 dead after crash at Miami Airport. And, Germany's far-right wins state election",Politics and Government; Business and Economy; Conflict and Security; Education; Environment and Climate; Health; Science; Society and Culture; Technology,True,Not reviewed


In [30]:
preliminary_reviews = {
    10: {
        "topics": "Conflict and Security; Politics and Government",
        "notes": "Other labels need verification against article content."
    },
    11: {
        "topics": "Education; Technology",
        "notes": "Health is possible but not clearly established by the headline."
    },
    12: {
        "topics": "Business and Economy; Politics and Government; Society and Culture",
        "notes": "Trade, governance, and human rights are relevant. Other labels need verification."
    },
    13: {
        "topics": "Society and Culture",
        "notes": "Immigration-related story. Politics may be relevant; full article needed."
    },
    14: {
        "topics": "Conflict and Security; Politics and Government",
        "notes": "Headline combines two news events. Other labels need verification."
    }
}

for idx, review in preliminary_reviews.items():
    review_sample.loc[idx, "reviewed_topics"] = review["topics"]
    review_sample.loc[idx, "review_notes"] = review["notes"]
    review_sample.loc[idx, "review_status"] = "Headline review only"

review_sample.to_csv(
    "../data/topic_review_sample.csv",
    index=False
)

print(
    review_sample["review_status"].value_counts()
)

review_status
Headline review only    15
Not reviewed            15
Name: count, dtype: int64


In [31]:
display(
    review_sample.loc[
        15:19,
        [
            "title",
            "topic",
            "needs_topic_review",
            "review_status"
        ]
    ]
)

,title,topic,needs_topic_review,review_status
15,Nonprofit claims ICE agent attempted to enter Cranberry Township home,Business and Economy; Conflict and Security; Environment and Climate; Politics and Government; Society and Culture,False,Not reviewed
16,Hotter winds threaten to fan flames of Europe's wildfires,Business and Economy; Conflict and Security; Education; Environment and Climate; Health; Politics and Government; Society and Culture,True,Not reviewed
17,"Thomas Gallatin: Hating Capitalism, but Loving Free Enterprise?",Business and Economy; Education; Environment and Climate; Health; Politics and Government; Science; Technology,True,Not reviewed
18,"SBI Research sees CPI inflation at 5.65% in Sept, above 6.5% in Oct-Nov",Business and Economy; Science; Conflict and Security; Environment and Climate; Politics and Government,False,Not reviewed
19,Fourteen Babies and the Republic of Pakistani Privilege,Business and Economy; Conflict and Security; Education; Environment and Climate; Health; Politics and Government; Society and Culture,True,Not reviewed


In [32]:
preliminary_reviews = {
    15: {
        "topics": "Politics and Government; Society and Culture",
        "notes": "Conflict and Security is possible. Other labels need verification."
    },
    16: {
        "topics": "Environment and Climate",
        "notes": "Health and Conflict and Security may be relevant depending on article content."
    },
    17: {
        "topics": "Business and Economy",
        "notes": "Politics and Government is possible. Other labels need verification."
    },
    18: {
        "topics": "Business and Economy",
        "notes": "Science may reflect the word Research rather than the article subject."
    },
    19: {
        "topics": "Society and Culture",
        "notes": "Headline is ambiguous. Full article needed before confirming topics."
    }
}

for idx, review in preliminary_reviews.items():
    review_sample.loc[idx, "reviewed_topics"] = review["topics"]
    review_sample.loc[idx, "review_notes"] = review["notes"]
    review_sample.loc[idx, "review_status"] = "Headline review only"

review_sample.to_csv(
    "../data/topic_review_sample.csv",
    index=False
)

print(
    review_sample["review_status"].value_counts()
)

review_status
Headline review only    20
Not reviewed            10
Name: count, dtype: int64


In [33]:
display(
    review_sample.loc[
        20:24,
        [
            "title",
            "topic",
            "needs_topic_review",
            "review_status"
        ]
    ]
)

,title,topic,needs_topic_review,review_status
20,Pentair Announces Bob Hau as Executive Vice President and Chief Financial Officer,Politics and Government; Business and Economy; Conflict and Security; Education; Environment and Climate; Science,False,Not reviewed
21,"Kevin Warsh does his job, not Trump's command (Editorial)",Business and Economy; Conflict and Security; Environment and Climate; Health; Politics and Government,False,Not reviewed
22,CanAlaska Uranium Ltd.: CanAlaska Announces Results from Nebula Winter Drill Program,Environment and Climate; Politics and Government; Technology,False,Not reviewed
23,Ringgit ends higher ahead of Fed chair Warsh's Jackson Hole speech,Business and Economy; Politics and Government,False,Not reviewed
24,Vienna-Leopoldstadt: New Bike Path Now Connects 2nd and 20th Districts Continuously,Business and Economy; Environment and Climate; Society and Culture,False,Not reviewed


In [34]:
preliminary_reviews = {
    20: {
        "topics": "Business and Economy",
        "notes": "Other assigned topics are not supported by the headline."
    },
    21: {
        "topics": "Politics and Government",
        "notes": "Business and Economy is possible. Full editorial needed."
    },
    22: {
        "topics": "Business and Economy",
        "notes": "Business is missing. Environment and Science are possible; full article needed."
    },
    23: {
        "topics": "Business and Economy",
        "notes": "Politics and Government may be relevant depending on article content."
    },
    24: {
        "topics": "Environment and Climate",
        "notes": "Society and Culture and Politics are possible. Business needs verification."
    }
}

for idx, review in preliminary_reviews.items():
    review_sample.loc[idx, "reviewed_topics"] = review["topics"]
    review_sample.loc[idx, "review_notes"] = review["notes"]
    review_sample.loc[idx, "review_status"] = "Headline review only"

review_sample.to_csv(
    "../data/topic_review_sample.csv",
    index=False
)

print(review_sample["review_status"].value_counts())

review_status
Headline review only    25
Not reviewed             5
Name: count, dtype: int64


In [35]:
display(
    review_sample.loc[
        25:29,
        [
            "title",
            "topic",
            "needs_topic_review",
            "review_status"
        ]
    ]
)

,title,topic,needs_topic_review,review_status
25,Metals Creek Resources Corp.: Metals Creek and Benton Complete Hydrogen-Helium Soil Gas Sampling at Smoking Gun and Parson's Pond in Newfoundland and Acquires Addition Ground,Conflict and Security; Business and Economy; Environment and Climate; Health; Politics and Government; Society and Culture; Technology,True,Not reviewed
26,Yemen's collapsing healthcare forces the sick to seek alternative medicines | Health News,Health; Business and Economy; Conflict and Security; Environment and Climate; Science; Society and Culture; Technology,True,Not reviewed
27,The copper peptide trend has gone from creams to injections — ahead of the science,Science; Business and Economy; Conflict and Security; Education; Health; Politics and Government; Society and Culture; Technology,True,Not reviewed
28,"Nell Nolan; Izzi Dennis Deb Party, Debs LeBlanc and Merington",Politics and Government; Society and Culture,False,Not reviewed
29,"Trump set to address UN amid war, AI anxiety and questions about world body's future",Conflict and Security; Technology; Business and Economy; Environment and Climate; Health; Politics and Government; Society and Culture,True,Not reviewed


In [36]:
preliminary_reviews = {
    25: {
        "topics": "Business and Economy",
        "notes": "Science and Environment are possible. Science is missing from existing labels."
    },
    26: {
        "topics": "Health; Conflict and Security",
        "notes": "Society and Culture is possible. Other labels need verification."
    },
    27: {
        "topics": "Health; Science",
        "notes": "Other assigned topics are not clearly supported by the headline."
    },
    28: {
        "topics": "Society and Culture",
        "notes": "Politics and Government appears unrelated."
    },
    29: {
        "topics": "Politics and Government; Conflict and Security; Technology",
        "notes": "Multiple topics are supported by the headline. Other labels need verification."
    }
}

for idx, review in preliminary_reviews.items():
    review_sample.loc[idx, "reviewed_topics"] = review["topics"]
    review_sample.loc[idx, "review_notes"] = review["notes"]
    review_sample.loc[idx, "review_status"] = "Headline review only"

review_sample.to_csv(
    "../data/topic_review_sample.csv",
    index=False
)

print(review_sample["review_status"].value_counts())

review_status
Headline review only    30
Name: count, dtype: int64


In [37]:
def compare_topics(row):
    original = set(row["topic"].split("; "))
    reviewed = set(row["reviewed_topics"].split("; "))

    return pd.Series({
        "original_count": len(original),
        "reviewed_count": len(reviewed),
        "overlapping_count": len(original & reviewed),
        "extra_count": len(original - reviewed),
        "missing_count": len(reviewed - original)
    })

topic_comparison = review_sample.apply(compare_topics, axis=1)

review_results = pd.concat(
    [review_sample, topic_comparison],
    axis=1
)

display(
    review_results.groupby("needs_topic_review")[
        [
            "original_count",
            "reviewed_count",
            "overlapping_count",
            "extra_count",
            "missing_count"
        ]
    ].mean().round(2)
)

,original_count,reviewed_count,overlapping_count,extra_count,missing_count
needs_topic_review,,,,,
False,3.6,1.20,1.07,2.53,0.13
True,7.4,1.67,1.67,5.73,0.00


In [38]:
def calculate_topic_scores(row):
    original = set(row["topic"].split("; "))
    reviewed = set(row["reviewed_topics"].split("; "))

    overlap = len(original & reviewed)

    precision = overlap / len(original) if original else 0
    recall = overlap / len(reviewed) if reviewed else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0
    )

    return pd.Series({
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

scores = review_sample.apply(calculate_topic_scores, axis=1)

print("Original topic labels vs. headline reviews")
print("Precision:", round(scores["precision"].mean(), 3))
print("Recall:", round(scores["recall"].mean(), 3))
print("F1:", round(scores["f1"].mean(), 3))

Original topic labels vs. headline reviews
Precision: 0.287
Recall: 0.933
F1: 0.414


In [39]:
review_sample["predicted_title_topics"] = (
    review_sample["title"].apply(get_title_topics)
)

display(
    review_sample[
        ["title", "reviewed_topics", "predicted_title_topics"]
    ].head(10)
)

,title,reviewed_topics,predicted_title_topics
0,Bank credit surge supports high GDP growth target: SBV,Business and Economy,[Business and Economy]
1,Experts Explain How To Tell If You Have A Calcium Deficiency,Health,[]
2,Donald Trump meets Irish president Catherine Connolly at start of two-day visit,Politics and Government,[Politics and Government]
3,Early Voting Begins Today in Miami-Dade; Broward Opens Sites Next Week,Politics and Government,[]
4,KITS Eyecare Reports Preliminary Third Quarter 2026 Results,Business and Economy; Health,[]
5,Zelenskyy says Ukraine received no intelligence from partners on incoming missile strike on Kyiv,Conflict and Security; Politics and Government,[Conflict and Security]
6,OSCP open-sources a ROS 2 driver for its MK2 inertial measurement units,Technology,[]
7,Cuban artist and dissident exiled from country after 5 years in prison arrives in US,Politics and Government; Society and Culture,[]
8,Chemistry Nobel Prize awarded to duo for research solving 'mirror image' mystery,Science,[Science]
9,Six-Year-Old Aditya and the Politics of a New Generation,Politics and Government,[Politics and Government]


In [40]:
def evaluate_title_topics(row):
    predicted = set(row["predicted_title_topics"])
    reviewed = set(row["reviewed_topics"].split("; "))

    overlap = len(predicted & reviewed)

    precision = overlap / len(predicted) if predicted else 0
    recall = overlap / len(reviewed) if reviewed else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0
    )

    return pd.Series({
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

title_scores = review_sample.apply(evaluate_title_topics, axis=1)

print("Headline classifier vs. headline reviews")
print("Precision:", round(title_scores["precision"].mean(), 3))
print("Recall:", round(title_scores["recall"].mean(), 3))
print("F1:", round(title_scores["f1"].mean(), 3))

Headline classifier vs. headline reviews
Precision: 0.389
Recall: 0.339
F1: 0.342


In [41]:
def get_hybrid_topics(row):
    title_topics = row["predicted_title_topics"]

    if title_topics:
        return title_topics

    return [row["primary_topic"]]


review_sample["hybrid_topics"] = review_sample.apply(
    get_hybrid_topics,
    axis=1
)

display(
    review_sample[
        [
            "title",
            "reviewed_topics",
            "predicted_title_topics",
            "hybrid_topics"
        ]
    ].head(10)
)

,title,reviewed_topics,predicted_title_topics,hybrid_topics
0,Bank credit surge supports high GDP growth target: SBV,Business and Economy,[Business and Economy],[Business and Economy]
1,Experts Explain How To Tell If You Have A Calcium Deficiency,Health,[],[Environment and Climate]
2,Donald Trump meets Irish president Catherine Connolly at start of two-day visit,Politics and Government,[Politics and Government],[Politics and Government]
3,Early Voting Begins Today in Miami-Dade; Broward Opens Sites Next Week,Politics and Government,[],[Politics and Government]
4,KITS Eyecare Reports Preliminary Third Quarter 2026 Results,Business and Economy; Health,[],[Business and Economy]
5,Zelenskyy says Ukraine received no intelligence from partners on incoming missile strike on Kyiv,Conflict and Security; Politics and Government,[Conflict and Security],[Conflict and Security]
6,OSCP open-sources a ROS 2 driver for its MK2 inertial measurement units,Technology,[],[Business and Economy]
7,Cuban artist and dissident exiled from country after 5 years in prison arrives in US,Politics and Government; Society and Culture,[],[Conflict and Security]
8,Chemistry Nobel Prize awarded to duo for research solving 'mirror image' mystery,Science,[Science],[Science]
9,Six-Year-Old Aditya and the Politics of a New Generation,Politics and Government,[Politics and Government],[Politics and Government]


In [42]:
def evaluate_hybrid(row):
    predicted = set(row["hybrid_topics"])
    reviewed = set(row["reviewed_topics"].split("; "))

    overlap = len(predicted & reviewed)

    precision = overlap / len(predicted) if predicted else 0
    recall = overlap / len(reviewed) if reviewed else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0
    )

    return pd.Series({
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

hybrid_scores = review_sample.apply(evaluate_hybrid, axis=1)

print("Hybrid classifier vs. headline reviews")
print("Precision:", round(hybrid_scores["precision"].mean(), 3))
print("Recall:", round(hybrid_scores["recall"].mean(), 3))
print("F1:", round(hybrid_scores["f1"].mean(), 3))

Hybrid classifier vs. headline reviews
Precision: 0.556
Recall: 0.467
F1: 0.481


In [43]:
# Final topic quality audit summary

audit_summary = pd.DataFrame({
    "Method": [
        "Original GDELT Labels",
        "Headline Classifier",
        "Hybrid Classifier"
    ],
    "Precision": [0.287, 0.389, 0.556],
    "Recall": [0.933, 0.339, 0.467],
    "F1": [0.414, 0.342, 0.481]
})

audit_summary.to_csv(
    "../data/topic_audit_summary.csv",
    index=False
)

print("Topic audit completed.")
print("Original dataset remains unchanged.")

display(audit_summary)

Topic audit completed.
Original dataset remains unchanged.


,Method,Precision,Recall,F1
0,Original GDELT Labels,0.287,0.933,0.414
1,Headline Classifier,0.389,0.339,0.342
2,Hybrid Classifier,0.556,0.467,0.481
